# Implement a Simple RAG System

## Document Chunking 

In [1]:
from typing import List

def split_into_chunks(doc_file: str) -> List[str]:
    with open(doc_file, 'r') as file:
        content = file.read()

    return [chunk for chunk in content.split("\n\n")]

chunks = split_into_chunks("doc_eng.md")

for i, chunk in enumerate(chunks):
    print(f"[{i}] {chunk}\n")

[0] # Doraemon and the Super Saiyan: Battle of Space and Time

[1] On an ordinary afternoon, Nobita was sitting at his desk staring blankly as usual. His homework was piled as high as a mountain, and he hadn't even started the first page. Doraemon was next to him, flipping through a comic book and sighing from time to time, feeling that this kid was as unreliable as ever. Just as their lives were carrying on normally, a bright light suddenly descended from the sky, shaking the entire room. Out of the light stepped a blond-haired boy wearing battle armor and exuding an astonishing aura; he was the Super Saiyan from the future—Trunks. As soon as he appeared, he spoke astonishing words: the Earth of the future was about to be destroyed by dark forces, and he had come to seek Doraemon's help.

[2] Doraemon and Nobita were shocked upon hearing this, but they also read an undeniable resolve in Trunks' determined eyes. Trunks explained that the enemy in the future was no ordinary villain, but

In [ ]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 100) -> List[str]:
    """
    Split text into overlapping chunks of specified size.
    
    Args:
        text: The text to chunk
        chunk_size: Maximum characters per chunk
        overlap: Number of characters to overlap between chunks
    
    Returns:
        List of text chunks
    """
    chunks = []
    start = 0
    
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end]
        chunks.append(chunk.strip())
        
        # Move start position by (chunk_size - overlap) for next iteration
        start += chunk_size - overlap
    
    return [c for c in chunks if c]  # Remove empty chunks


def chunk_by_paragraphs(text: str, max_chunk_size: int = 500) -> List[str]:
    """
    Split text into chunks while preserving paragraph boundaries.
    
    Args:
        text: The text to chunk
        max_chunk_size: Maximum characters per chunk
    
    Returns:
        List of text chunks
    """
    paragraphs = text.split("\n\n")
    chunks = []
    current_chunk = ""
    
    for paragraph in paragraphs:
        if len(current_chunk) + len(paragraph) <= max_chunk_size:
            current_chunk += paragraph + "\n\n"
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = paragraph + "\n\n"
    
    if current_chunk:
        chunks.append(current_chunk.strip())
    
    return chunks


def chunk_by_sentences(text: str, max_chunk_size: int = 500) -> List[str]:
    """
    Split text into chunks while preserving sentence boundaries.
    Requires sentence_transformers or nltk for sentence tokenization.
    
    Args:
        text: The text to chunk
        max_chunk_size: Maximum characters per chunk
    
    Returns:
        List of text chunks
    """
    import re
    # Simple sentence split by common punctuation
    sentences = re.split(r'(?<=[。！？\.\!\?])\s+', text)
    chunks = []
    current_chunk = ""
    
    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= max_chunk_size:
            current_chunk += sentence + " "
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sentence + " "
    
    if current_chunk:
        chunks.append(current_chunk.strip())
    
    return chunks


# Example usage:
sample_text = """This is a long text that needs to be split into chunks.
Each chunk should be manageable in size for processing.

This is another paragraph that will be included in the chunking process.
The chunking function should handle this gracefully."""

# Test different chunking methods
print("=== Character-based chunking (size=100, overlap=20) ===")
char_chunks = chunk_text(sample_text, chunk_size=100, overlap=20)
for i, chunk in enumerate(char_chunks):
    print(f"[{i}] {chunk[:50]}... (len={len(chunk)})\n")

print("\n=== Paragraph-aware chunking (max_size=200) ===")
para_chunks = chunk_by_paragraphs(sample_text, max_chunk_size=200)
for i, chunk in enumerate(para_chunks):
    print(f"[{i}] {chunk[:50]}... (len={len(chunk)})\n")

## Embedding Model

In [6]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
#embedding_model = SentenceTransformer("shibing624/text2vec-base-chinese")
#embedding_model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

def embed_chunk(chunk: str) -> List[float]:
    embedding = embedding_model.encode(chunk)

    return embedding.tolist()

test_embedding = embed_chunk("testing: Saint Seiya")
print(len(test_embedding))
print(test_embedding)

Loading weights: 100%|████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 6012.97it/s]


768
[-0.010587815195322037, 0.02579634264111519, -0.052153728902339935, 0.03141562268137932, -0.023490959778428078, -0.005997709929943085, 0.018270689994096756, 0.02386787161231041, 0.027359042316675186, 0.03427219018340111, 0.07386306673288345, 0.030300287529826164, 0.011995099484920502, 0.06931719928979874, 0.023803265765309334, 0.03644938766956329, 0.014438490383327007, -0.019937630742788315, -0.0406421422958374, 0.018973935395479202, -0.03488747775554657, -0.026887133717536926, -0.01874539442360401, -0.015495914965867996, 0.01717551052570343, 0.0051718829199671745, -0.0338347963988781, 3.631051004049368e-05, -0.01567487232387066, -0.0661725327372551, -0.004762138705700636, 0.04489727318286896, 0.0181149709969759, 0.026409460231661797, 1.8238373513668193e-06, -0.04224328696727753, 0.0638737827539444, -0.024251695722341537, -0.04208717122673988, 0.022183336317539215, -0.06724179536104202, 0.0676620751619339, -0.010663195513188839, 0.019243881106376648, 0.0005067926249466836, -0.01065

In [7]:
embeddings = [embed_chunk(chunk) for chunk in chunks]

print(len(embeddings))
print(len(embeddings[0]))

10
768


## Vector Database (Storage)

In [9]:
import chromadb

# Creates an in-memory database client (EphemeralClient) that stores data temporarily in RAM
chromadb_client = chromadb.EphemeralClient()

# Gets or creates a collection named "default" where embeddings will be stored
chromadb_collection = chromadb_client.get_or_create_collection(name="default")

def save_embeddings(chunks: List[str], embeddings: List[List[float]]) -> None:
    ids = [str(i) for i in range(len(chunks))]
    chromadb_collection.add(
        documents=chunks,
        embeddings=embeddings,
        ids=ids
    )

save_embeddings(chunks, embeddings)

## Retrieve

In [10]:
def retrieve(query: str, top_k: int) -> List[str]:
    # embedding the query
    query_embedding = embed_chunk(query)
    
    results = chromadb_collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    return results['documents'][0]

query = "What were the three secret gadgets used by Doraemon? "

retrieve_chunks = retrieve(query, 5)

for i, chunk in enumerate(retrieve_chunks):
    print(f"[{i}] {chunk}\n")

[0] The three secret gadgets were: The "Copy Cape," which could temporarily grant super combat power. The "Time-Stopping Watch," which could pause time for five seconds. The "Portable Hyperbolic Time Chamber," which allowed one to complete a year of training in just one minute. Nobita was pushed into the chamber, undergoing intensive training inside. Although only a few minutes passed in real time, he experienced a full year of rigorous ascetic training. At first, he was still weak, wanting to give up and run away, but when he thought of Shizuka, his parents, and Doraemon's determined gaze, he finally gritted his teeth and persevered. After coming out, his body and mind were completely renewed, and his eyes held a new level of maturity and confidence.

[1] And so, Doraemon, taking Trunks and Nobita along, activated the Time Machine and traveled to that collapsing future world. The sight before them was shocking: cities were reduced to ruins, the ground was crisscrossed with fissures, a

## Re-Ranking

In [11]:
from sentence_transformers import CrossEncoder

def rerank(query: str, retrieved_chunks: List[str], top_k: int) -> List[str]:
    # Loads a multilingual re-ranker model (mmarco-mMiniLMv2-L12-H384-v1) that scores query-document pairs for relevance
    cross_encoder = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1')

    # Creates (query, chunk) tuples for each retrieved chunk to compare relevance
    pairs = [(query, chunk) for chunk in retrieved_chunks]

    # Uses CrossEncoder to predict relevance scores for each pair (higher score = more relevant)
    scores = cross_encoder.predict(pairs)

    chunk_with_score = [(x, y) for x, y in  zip(retrieved_chunks, scores)]
    chunk_with_score.sort(key=lambda pair: pair[1], reverse=True)

    return [chunk for chunk, _ in chunk_with_score][:top_k]

reranked_chunks = rerank(query, retrieve_chunks, 3)

for i, chunk in enumerate(reranked_chunks):
    print(f"[{i}] {chunk}\n")

Loading weights: 100%|████████████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 7700.89it/s]


[0] The three secret gadgets were: The "Copy Cape," which could temporarily grant super combat power. The "Time-Stopping Watch," which could pause time for five seconds. The "Portable Hyperbolic Time Chamber," which allowed one to complete a year of training in just one minute. Nobita was pushed into the chamber, undergoing intensive training inside. Although only a few minutes passed in real time, he experienced a full year of rigorous ascetic training. At first, he was still weak, wanting to give up and run away, but when he thought of Shizuka, his parents, and Doraemon's determined gaze, he finally gritted his teeth and persevered. After coming out, his body and mind were completely renewed, and his eyes held a new level of maturity and confidence.

[1] And so, Doraemon, taking Trunks and Nobita along, activated the Time Machine and traveled to that collapsing future world. The sight before them was shocking: cities were reduced to ruins, the ground was crisscrossed with fissures, a

## Generate

In [16]:
from dotenv import load_dotenv
from google import genai
from pathlib import Path

env_path = Path('.').resolve() / '.env'
load_dotenv(dotenv_path=env_path, override=True)
google_client = genai.Client()

def generate(query: str, chunks: List[str]) -> str:
    sep = "\n\n"
    
    prompt = f"""You are a knowledgeable assistant. 
Please generate an accurate response based on the user's questions and the following snippets.
    
Question: {query}

Snippets:
    {sep.join(chunks)}

Please answer based on the content above and do not fabricate information.
    """

    print(f"{prompt} {sep}")
    print('-'*100)

    response = google_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

answer = generate(query, reranked_chunks)
print(answer)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


You are a knowledgeable assistant. 
Please generate an accurate response based on the user's questions and the following snippets.

Question: What were the three secret gadgets used by Doraemon? 

Snippets:
    The three secret gadgets were: The "Copy Cape," which could temporarily grant super combat power. The "Time-Stopping Watch," which could pause time for five seconds. The "Portable Hyperbolic Time Chamber," which allowed one to complete a year of training in just one minute. Nobita was pushed into the chamber, undergoing intensive training inside. Although only a few minutes passed in real time, he experienced a full year of rigorous ascetic training. At first, he was still weak, wanting to give up and run away, but when he thought of Shizuka, his parents, and Doraemon's determined gaze, he finally gritted his teeth and persevered. After coming out, his body and mind were completely renewed, and his eyes held a new level of maturity and confidence.

And so, Doraemon, taking Trunk

## Use Google Cloud Natural Language API

In [ ]:
def analyze_sentiment_with_gemini(text: str) -> dict:
    """
    Analyze sentiment of text using Gemini API.
    
    Args:
        text: The text to analyze
    
    Returns:
        Dictionary with sentiment scores and analysis
    """
    prompt = f"""分析以下文本的情感。请返回JSON格式的结果，包含以下字段：
- sentiment: positive/neutral/negative
- confidence: 0-1 的置信度
- explanation: 简短的情感分析说明

文本: {text}

请只返回JSON，不要其他内容。"""
    
    response = google_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    
    import json
    try:
        result = json.loads(response.text)
        return result
    except:
        return {"sentiment": "unknown", "raw_response": response.text}


# Usage
result = analyze_sentiment_with_gemini("这个产品太好了，我非常喜欢！")
print(result)

In [ ]:
def analyze_sentiments_batch(chunks: List[str]) -> List[dict]:
    """
    Analyze sentiment for multiple text chunks.
    
    Args:
        chunks: List of text chunks to analyze
    
    Returns:
        List of sentiment analysis results
    """
    results = []
    for i, chunk in enumerate(chunks):
        try:
            sentiment = analyze_sentiment_with_gemini(chunk)
            sentiment['chunk_id'] = i
            results.append(sentiment)
        except Exception as e:
            results.append({'chunk_id': i, 'error': str(e)})
    
    return results

# Usage with your existing chunks
sentiments = analyze_sentiments_batch(chunks)

In [4]:
!pip list | grep google-cloud-language

google-cloud-language             2.20.0


In [2]:
!pip install google-cloud-language

  Using cached google_api_core-2.30.3-py3-none-any.whl.metadata (3.1 kB)
  Using cached google_auth-2.49.2-py3-none-any.whl.metadata (6.2 kB)
  Using cached proto_plus-1.27.2-py3-none-any.whl.metadata (2.2 kB)
  Using cached googleapis_common_protos-1.74.0-py3-none-any.whl.metadata (9.2 kB)
  Using cached grpcio_status-1.80.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached protobuf-6.33.6-cp39-abi3-macosx_10_9_universal2.whl.metadata (593 bytes)
Using cached google_api_core-2.30.3-py3-none-any.whl (173 kB)
Using cached google_auth-2.49.2-py3-none-any.whl (240 kB)
Using cached googleapis_common_protos-1.74.0-py3-none-any.whl (300 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 46.8 MB/s eta 0:00:00 0:00:01
Using cached grpcio_status-1.80.0-py3-none-any.whl (14 kB)
Using cached proto_plus-1.27.2-py3-none-any.whl (50 kB)
Using cached protobuf-6.33.6-cp39-abi3-macosx_10_9_universal2.whl (427 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.

In [5]:
from google.cloud import language_v1

## Test GCP Natural Language API setup
def test_gcp_setup():
    try:
        client = language_v1.LanguageServiceClient()
        print("✅ GCP Natural Language API setup successful!")
        return True
    except Exception as e:
        print(f"❌ Setup failed: {e}")
        return False

test_gcp_setup()

ImportError: cannot import name 'language_v1' from 'google.cloud' (unknown location)

In [ ]:
from google.cloud import language_v1

def analyze_sentiment_with_gcp(text: str) -> dict:
    """
    Analyze sentiment using Google Cloud Natural Language API.
    
    Args:
        text: The text to analyze
    
    Returns:
        Dictionary with sentiment score and magnitude
    """
    client = language_v1.LanguageServiceClient()
    document = language_v1.Document(
        content=text,
        type_=language_v1.Document.Type.PLAIN_TEXT,
        language="zh"  # Chinese
    )
    
    sentiment = client.analyze_sentiment(request={'document': document})
    
    return {
        "score": sentiment.document_sentiment.score,  # -1.0 to 1.0
        "magnitude": sentiment.document_sentiment.magnitude,  # Strength of emotion
        "sentences": [
            {
                "text": sentence.text.content,
                "score": sentence.sentiment.score,
                "magnitude": sentence.sentiment.magnitude
            }
            for sentence in sentiment.sentences
        ]
    }